# Phase 2 attention-intervention API

This notebook demonstrates the existing `research.phase2.intervention` API. It does **not** change a model architecture, checkpoint, or evaluator.

> **Important:** the default tiny, randomly initialized CPU model demonstrates intervention mechanics only. It does not demonstrate pretrained tool-selection behavior, model quality, or an experimental result.

The default executes entirely on CPU and does not download weights. Configuration-only choices report architecture-derived layer IDs and expected attention parameter shapes; those are not a substitute for inspecting a loaded model.

In [ ]:
# Edit these values first. No checkpoint is loaded unless LOAD_CHECKPOINT is True.
MODEL_CHOICE = "tiny_cpu"
# Choices: tiny_cpu, llama31_config_only, gpt_oss_config_only, loaded_model, llama31_checkpoint
CHECKPOINT_DIR = None  # Optional local directory, for example Path("/path/to/checkpoint")
LOAD_CHECKPOINT = False  # Required for the llama31_checkpoint choice. Never downloads weights.
LOADED_MODEL = None  # Set this to a model you already loaded for loaded_model.
LAYER_INDEX = 0
RUN_ZERO_STRENGTH_ACCEPTANCE_PROBE = True  # Exact API check; restores before this cell ends.

assert MODEL_CHOICE in {
    "tiny_cpu", "llama31_config_only", "gpt_oss_config_only",
    "loaded_model", "llama31_checkpoint",
}


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import torch
from torch import nn

from research.phase2.intervention import (
    AttentionInterventionError,
    attention_module_for_layer,
    available_attention_layers,
    temporary_attention_intervention,
)


In [ ]:
# A deliberately tiny supported layout for the CPU-only mechanics demonstration.
class TinyDecoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.self_attn = nn.Linear(4, 4)
        self.mlp = nn.Linear(4, 4)


class TinySupportedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([TinyDecoderLayer(), TinyDecoderLayer()])


def parameter_table(model, layer_index):
    module = attention_module_for_layer(model, layer_index)
    return [
        {
            "name": name,
            "shape": tuple(parameter.shape),
            "dtype": str(parameter.dtype),
            "device": str(parameter.device),
        }
        for name, parameter in module.named_parameters(recurse=True)
    ]


def llama31_config_view():
    # The repository's local Llama loader uses these built-in architecture defaults.
    from models.architectures.llama31_8b_pytorch.model import ModelConfigs

    cfg = ModelConfigs()
    q = cfg.num_attention_heads * cfg.head_dim
    kv = cfg.num_key_value_heads * cfg.head_dim
    rows = [
        ("q_proj.weight", (q, cfg.hidden_size)),
        ("k_proj.weight", (kv, cfg.hidden_size)),
        ("v_proj.weight", (kv, cfg.hidden_size)),
        ("o_proj.weight", (cfg.hidden_size, q)),
    ]
    if cfg.attention_bias:
        rows.extend([("q_proj.bias", (q,)), ("k_proj.bias", (kv,)),
                     ("v_proj.bias", (kv,)), ("o_proj.bias", (cfg.hidden_size,))])
    return tuple(range(cfg.num_hidden_layers)), "model.layers[i].self_attn", rows


def gpt_oss_config_view(checkpoint_dir):
    from models.architectures.gpt_oss_pytorch.model import ModelConfigs

    config_path = Path(checkpoint_dir) / "config.json" if checkpoint_dir else None
    if config_path and config_path.is_file():
        cfg = ModelConfigs(**json.loads(config_path.read_text()))
        source = f"local metadata: {config_path}"
    else:
        cfg = ModelConfigs()
        source = "repository defaults (no checkpoint metadata read)"
    qkv = cfg.head_dim * (cfg.num_attention_heads + 2 * cfg.num_key_value_heads)
    rows = [
        ("sinks", (cfg.num_attention_heads,)),
        ("norm.scale", (cfg.hidden_size,)),
        ("qkv.weight", (qkv, cfg.hidden_size)),
        ("qkv.bias", (qkv,)),
        ("out.weight", (cfg.hidden_size, cfg.num_attention_heads * cfg.head_dim)),
        ("out.bias", (cfg.hidden_size,)),
    ]
    return tuple(range(cfg.num_hidden_layers)), "model.block[i].attn", rows, source


In [ ]:
# Inspect an actual supplied model, or inspect expected architecture shapes without weights.
checkpoint_path = Path(CHECKPOINT_DIR) if CHECKPOINT_DIR is not None else None
active_model = None

if MODEL_CHOICE == "tiny_cpu":
    torch.manual_seed(0)
    active_model = TinySupportedModel().cpu()
elif MODEL_CHOICE == "loaded_model":
    if LOADED_MODEL is None:
        raise ValueError("Set LOADED_MODEL to an already-loaded torch model.")
    active_model = LOADED_MODEL
elif MODEL_CHOICE == "llama31_checkpoint":
    if not LOAD_CHECKPOINT or checkpoint_path is None:
        raise ValueError("llama31_checkpoint requires CHECKPOINT_DIR and LOAD_CHECKPOINT = True.")
    if not checkpoint_path.is_dir():
        raise FileNotFoundError(checkpoint_path)
    from models.architectures.llama31_8b_pytorch.config import Config
    from models.architectures.llama31_8b_pytorch.inference import TokenGenerator
    active_model = TokenGenerator(checkpoint=str(checkpoint_path), device=Config.device).model

if active_model is not None:
    layer_ids = available_attention_layers(active_model)
    if LAYER_INDEX not in layer_ids:
        raise ValueError(f"LAYER_INDEX={LAYER_INDEX}; valid IDs are {layer_ids}")
    attention = attention_module_for_layer(active_model, LAYER_INDEX)
    print("Actual loaded/tiny model")
    print("available layer IDs:", layer_ids)
    print("selected attention module:", type(attention).__qualname__)
    print("selected attention parameters:")
    for row in parameter_table(active_model, LAYER_INDEX):
        print("  " + str(row))
elif MODEL_CHOICE == "llama31_config_only":
    layer_ids, module_name, rows = llama31_config_view()
    print("Configuration-only view; no model or weights were loaded.")
    print("expected layer IDs:", layer_ids)
    print("expected attention module:", module_name)
    print("expected parameter shapes:", rows)
elif MODEL_CHOICE == "gpt_oss_config_only":
    layer_ids, module_name, rows, source = gpt_oss_config_view(checkpoint_path)
    print("Configuration-only view; no model or weights were loaded.")
    print("configuration source:", source)
    print("expected layer IDs:", layer_ids)
    print("expected attention module:", module_name)
    print("expected parameter shapes:", rows)


## Quantization metadata versus actual selected-attention parameter types

Checkpoint JSON can describe how weights were stored, but it does not prove the types of the tensors that the runtime loaded. The API checks both model-level `is_loaded_in_4bit` / `is_loaded_in_8bit` flags and the actual selected attention parameters. It requires floating-point parameters and rejects either model-level flag. The local GPT-OSS runtime dequantizes its MXFP4 MLP checkpoint tensors while loading; that metadata alone does not determine the selected attention parameter dtype. Inspect the loaded model below.

In [ ]:
def quantization_metadata(path):
    if path is None:
        return {"status": "no checkpoint directory supplied"}
    if not path.is_dir():
        return {"status": f"not a directory: {path}"}
    findings = {}
    for filename in ("config.json", "quantization_config.json", "quantize_config.json"):
        candidate = path / filename
        if not candidate.is_file():
            continue
        try:
            payload = json.loads(candidate.read_text())
        except json.JSONDecodeError:
            findings[filename] = "not valid JSON"
            continue
        quantized = {key: value for key, value in payload.items() if "quant" in key.lower()}
        if quantized:
            findings[filename] = quantized
    return findings or {"status": "no top-level quantization metadata found"}


print("checkpoint metadata (storage description, not runtime proof):")
print(quantization_metadata(checkpoint_path))

if active_model is None:
    print("No loaded model: actual attention dtype and API acceptance cannot be established.")
else:
    attention_parameters = list(attention.named_parameters(recurse=True))
    model_flags = {
        name: bool(getattr(active_model, name, False))
        for name in ("is_loaded_in_4bit", "is_loaded_in_8bit")
    }
    print("model-level quantization flags:", model_flags)
    print("actual selected-attention types:",
          [(name, str(parameter.dtype), parameter.is_floating_point())
           for name, parameter in attention_parameters])
    preliminary_ok = not any(model_flags.values()) and bool(attention_parameters) and all(
        parameter.is_floating_point() for _, parameter in attention_parameters
    )
    print("preliminary current-API result:",
          "candidate for acceptance" if preliminary_ok else "will be rejected")
    if RUN_ZERO_STRENGTH_ACCEPTANCE_PROBE:
        try:
            # This invokes the public API's full structural checks. Strength zero preserves values.
            with temporary_attention_intervention(
                active_model, LAYER_INDEX, method="noise", strength=0.0, seed=0
            ) as probe:
                print("full current-API result: accepted; changed names:", probe.changed_parameter_names)
            print("exactly restored after probe:", probe.restored_exactly)
        except AttentionInterventionError as error:
            print("full current-API result: rejected:", error)


## Tiny CPU mechanics demonstration

Again, this is a randomly initialized model. It demonstrates that `enabled=False` is a no-op, that `noise` and `replace` affect only one selected attention module while active, and that all original values are restored exactly. It does not make an inference about a pretrained model or tool selection.

In [ ]:
def clone_parameters(model):
    return {name: parameter.detach().clone() for name, parameter in model.named_parameters()}


def exact_match(model, saved):
    return all(torch.equal(parameter, saved[name]) for name, parameter in model.named_parameters())


torch.manual_seed(123)
demo_model = TinySupportedModel().cpu()
demo_layer = 1
before = clone_parameters(demo_model)

# Disabled: neither weights nor output change.
inputs = torch.ones(2, 4)
baseline = demo_model.layers[demo_layer].self_attn(inputs).detach().clone()
with temporary_attention_intervention(
    demo_model, demo_layer, method="noise", strength=0.5, seed=7, enabled=False
) as disabled:
    disabled_output = demo_model.layers[demo_layer].self_attn(inputs).detach().clone()
assert torch.equal(baseline, disabled_output)
assert disabled.changed_parameter_names == ()
assert exact_match(demo_model, before)

results = {"disabled": "no-op; exact values retained"}
for method in ("noise", "replace"):
    with temporary_attention_intervention(
        demo_model, demo_layer, method=method, strength=0.5, seed=7, enabled=True
    ) as intervention:
        active = clone_parameters(demo_model)
        assert intervention.changed_parameter_names
        for name, value in active.items():
            targeted = name.startswith(f"layers.{demo_layer}.self_attn.")
            assert torch.equal(value, before[name]) is (not targeted), name
        results[method] = {
            "changed_attention_parameters": intervention.changed_parameter_names,
        }
    assert intervention.restored_exactly
    assert exact_match(demo_model, before)
    results[method]["restored_after_context"] = intervention.restored_exactly

print(results)


For a real loaded model, choose `loaded_model` and set `LOADED_MODEL`, inspect the reported valid IDs, then use `temporary_attention_intervention(model, layer_index, method="noise" or `"replace"`, strength=..., seed=..., enabled=...)` around the existing inference call. Use `enabled=False` for the unchanged call at the same call site. This notebook intentionally does not submit jobs, run an evaluator, save modified weights, or perform a layer sweep.